# 모델 구성 및 학습 연산 검증

DINOv3 블록을 축소된 규격으로 구성하여 CPDINO의 forward, 손실 계산 및 파라미터 갱신을 CPU에서 검증한다. 합성 입력에 대한 동작 검증이며 분할 정확도를 평가하지 않는다. 공식 ViT-S 가중치 초기화와 데이터 기반 학습은 `02_training.ipynb`에서 수행한다. 실행 환경은 README의 설치 절차를 따른다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository'
sys.path.insert(0, str(ROOT / 'src'))
import torch
torch.set_num_threads(2)
torch.manual_seed(20261112)
print('Repository', ROOT.name, '| PyTorch', torch.__version__)

## 1. DINOv3 인코더와 CPDINO 출력 헤드 구성

In [ ]:
from dinov3.models.vision_transformer import DinoVisionTransformer
from cpdino_lab.models import adapt_encoder, build_model
encoder = DinoVisionTransformer(img_size=32, patch_size=16, embed_dim=48, depth=2,
    num_heads=3, n_storage_tokens=4, layerscale_init=1e-5, pos_embed_rope_dtype='fp32')
encoder.init_weights()
student = adapt_encoder(encoder, 'unit-demo', rdrop=0)
images = torch.rand(1, 3, 32, 48)
labels = torch.zeros(1, 3, 32, 48)
prediction, _ = student(images)
assert prediction.shape == (1, 3, 32, 48)
print('Output', tuple(prediction.shape))

## 2. ViT-L, ViT-B 및 ViT-S의 구조

Meta device에서 모델을 구성하여 파라미터 메모리를 할당하지 않고 블록 수, embedding dimension 및 파라미터 수를 비교한다.

In [ ]:
for name in ['vitl','vitb','vits']:
    with torch.device('meta'):
        model = build_model(name)
    print(name, 'blocks', len(model.encoder.blocks), 'readout', tuple(model.out.weight.shape),
          'encoder parameters', sum(p.numel() for p in model.encoder.parameters()))
del model

## 3. 지도 손실과 출력 증류 손실

In [ ]:
import copy
from cpdino_lab.losses import segmentation_loss
teacher = copy.deepcopy(student).eval()
with torch.no_grad():
    teacher.out.bias.add_(0.5)
    target = teacher(images)[0]
task = segmentation_loss(prediction, labels)
distilled = segmentation_loss(prediction, labels, target, teacher_weight=1)
assert distilled > task
print({'supervised_loss':float(task.detach()), 'with_distillation':float(distilled.detach())})

## 4. Student 파라미터 갱신

한 번의 optimizer step 이후 student 출력 헤드의 변화와 teacher에 대한 gradient 차단을 검사한다.

In [ ]:
from cpdino_lab.engine import train_step, evaluate
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3)
before = student.out.weight.detach().clone()
loss = train_step(student, optimizer, images, labels, teacher)
assert not torch.equal(before, student.out.weight)
assert all(p.grad is None for p in teacher.parameters())
print('Training loss', loss, '| validation', evaluate(student, [(images, labels)]))

## 5. 체크포인트 저장 및 상태 복원

In [ ]:
from cpdino_lab.checkpoints import save_checkpoint, restore_checkpoint
checkpoint = ROOT / 'runs/unit-demo/last.pt'
identity = {'experiment':'unit-demo', 'seed':20261112}
save_checkpoint(checkpoint, student, optimizer, epoch=1, identity=identity)
epoch = restore_checkpoint(checkpoint, student, optimizer, identity)
assert epoch == 1
print('Restored epoch', epoch)

## 6. 모듈별 단위 테스트

In [ ]:
import pytest
result = pytest.main([str(ROOT / 'tests'), '-q'])
assert result == pytest.ExitCode.OK